# EnergyPredict AI — Ingeniería de Características

## 1. Introducción

### Objetivo

El objetivo de este notebook es transformar la serie temporal de demanda eléctrica española de 2024 en un conjunto de datos adecuado para entrenar modelos de Machine Learning.

Se desarrollarán variables predictoras (*features*) basadas en:

- Información temporal y de calendario.
- Valores históricos de demanda eléctrica.
- Patrones diarios y semanales.
- Estadísticas móviles de la serie temporal.

### Problema predictivo

Se plantea un problema de **regresión supervisada**, cuyo objetivo es predecir la demanda eléctrica media de la siguiente hora, expresada en megavatios (MW).

Para cada instante de predicción se utilizará únicamente información disponible hasta ese momento, evitando la incorporación accidental de información futura (*data leakage*).

### Dataset de partida

Se utiliza el archivo `demand_spain_2024.parquet`, generado durante la fase ETL.

Este conjunto contiene 8.784 observaciones horarias correspondientes a 2024, incluyendo una hora con un valor reconstruido parcialmente mediante interpolación.

## 2. Carga y preparación de los datos

Se carga el dataset procesado y se ordenan cronológicamente las observaciones.

Para formular la predicción de la siguiente hora, se utilizará como variable objetivo la demanda del instante posterior.

La separación cronológica es fundamental, ya que los modelos de series temporales deben evaluarse utilizando observaciones posteriores a las empleadas durante el entrenamiento.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar dataset procesado
df = pd.read_parquet(
    "../data/processed/demand_spain_2024.parquet"
)

# Ordenar cronológicamente
df = df.sort_values("datetime_utc").reset_index(drop=True)

# Comprobar estructura
print("Dimensiones:", df.shape)
print("\nColumnas:", df.columns.tolist())
print("\nPrimeros registros:")
display(df.head())

# Verificar continuidad temporal
assert df["datetime_utc"].diff().dropna().eq(
    pd.Timedelta(hours=1)
).all(), "La serie no tiene intervalos horarios consecutivos."

print("\nSerie temporal horaria validada.")


Dimensiones: (8784, 3)

Columnas: ['datetime_utc', 'demand_mw', 'contains_interpolation']

Primeros registros:


,datetime_utc,demand_mw,contains_interpolation
0,2024-01-01 00:00:00+00:00,21378.0,False
1,2024-01-01 01:00:00+00:00,20286.0,False
2,2024-01-01 02:00:00+00:00,19074.0,False
3,2024-01-01 03:00:00+00:00,18261.0,False
4,2024-01-01 04:00:00+00:00,17899.0,False



Serie temporal horaria validada.


## 3. Definición de la variable objetivo

La variable objetivo (*target*) será `target_next_hour`, que representa la demanda eléctrica de la hora inmediatamente posterior.

Se obtiene desplazando la columna `demand_mw` una posición hacia atrás mediante `shift(-1)`.

De esta forma, cada observación contiene la información disponible en la hora actual y el valor que el modelo deberá aprender a predecir.

La última observación no tendrá un valor objetivo conocido dentro del dataset, por lo que se excluirá posteriormente del entrenamiento.

In [2]:
# Crear variable objetivo: demanda de la siguiente hora
df["target_next_hour"] = df["demand_mw"].shift(-1)

# Fecha correspondiente a la predicción
df["target_datetime_utc"] = (
    df["datetime_utc"] + pd.Timedelta(hours=1)
)

# Visualizar el desplazamiento
display(
    df[
        [
            "datetime_utc",
            "demand_mw",
            "target_datetime_utc",
            "target_next_hour"
        ]
    ].head(10)
)

print(
    "Valores objetivo ausentes:",
    df["target_next_hour"].isna().sum()
)


,datetime_utc,demand_mw,target_datetime_utc,target_next_hour
0,2024-01-01 00:00:00+00:00,21378.0,2024-01-01 01:00:00+00:00,20286.0
1,2024-01-01 01:00:00+00:00,20286.0,2024-01-01 02:00:00+00:00,19074.0
2,2024-01-01 02:00:00+00:00,19074.0,2024-01-01 03:00:00+00:00,18261.0
3,2024-01-01 03:00:00+00:00,18261.0,2024-01-01 04:00:00+00:00,17899.0
4,2024-01-01 04:00:00+00:00,17899.0,2024-01-01 05:00:00+00:00,17992.0
5,2024-01-01 05:00:00+00:00,17992.0,2024-01-01 06:00:00+00:00,18363.0
6,2024-01-01 06:00:00+00:00,18363.0,2024-01-01 07:00:00+00:00,18477.0
7,2024-01-01 07:00:00+00:00,18477.0,2024-01-01 08:00:00+00:00,18596.0
8,2024-01-01 08:00:00+00:00,18596.0,2024-01-01 09:00:00+00:00,19548.0
9,2024-01-01 09:00:00+00:00,19548.0,2024-01-01 10:00:00+00:00,20895.0


Valores objetivo ausentes: 1


## 4. Creación de variables temporales

Los resultados del análisis exploratorio mostraron que la demanda eléctrica presenta patrones horarios, semanales y estacionales.

Por ello, se crearán variables de calendario correspondientes al instante que se desea predecir.

### Variables generadas

- `hour`: hora del día.
- `day_of_week`: día de la semana, de 0 (lunes) a 6 (domingo).
- `month`: mes del año.
- `is_weekend`: indicador binario de fin de semana.

Las fechas se convertirán al horario peninsular español mediante `Europe/Madrid`, respetando los cambios entre horario de invierno y verano.

Estas variables son conocidas de antemano y pueden utilizarse sin introducir información futura sobre la demanda.

In [3]:
# Convertir la fecha objetivo a horario peninsular
target_local = df["target_datetime_utc"].dt.tz_convert(
    "Europe/Madrid"
)

# Variables de calendario de la hora a predecir
df["hour"] = target_local.dt.hour
df["day_of_week"] = target_local.dt.dayofweek
df["month"] = target_local.dt.month

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

# Inspeccionar las nuevas características
display(
    df[
        [
            "target_datetime_utc",
            "hour",
            "day_of_week",
            "month",
            "is_weekend",
            "target_next_hour"
        ]
    ].head(10)
)


/tmp/ipykernel_62470/562253136.py:8: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = target_local.dt.dayofweek


,target_datetime_utc,hour,day_of_week,month,is_weekend,target_next_hour
0,2024-01-01 01:00:00+00:00,2,0,1,0,20286.0
1,2024-01-01 02:00:00+00:00,3,0,1,0,19074.0
2,2024-01-01 03:00:00+00:00,4,0,1,0,18261.0
3,2024-01-01 04:00:00+00:00,5,0,1,0,17899.0
4,2024-01-01 05:00:00+00:00,6,0,1,0,17992.0
5,2024-01-01 06:00:00+00:00,7,0,1,0,18363.0
6,2024-01-01 07:00:00+00:00,8,0,1,0,18477.0
7,2024-01-01 08:00:00+00:00,9,0,1,0,18596.0
8,2024-01-01 09:00:00+00:00,10,0,1,0,19548.0
9,2024-01-01 10:00:00+00:00,11,0,1,0,20895.0


## 5. Creación de variables históricas (Lags)

### Objetivo

Incorporar información sobre la demanda eléctrica registrada en horas anteriores para mejorar la capacidad predictiva del modelo.

Durante el análisis exploratorio se identificaron correlaciones elevadas entre la demanda actual y la registrada una hora, un día y una semana antes.

### Variables generadas

- `lag_1`: demanda de la hora actual, última observación disponible.
- `lag_2`: demanda registrada una hora antes de la actual.
- `lag_24`: demanda registrada 24 horas antes de la hora que se desea predecir.
- `lag_168`: demanda registrada 168 horas antes de la hora que se desea predecir.

### Prevención del Data Leakage

Los retardos se calcularán tomando como referencia la hora objetivo, utilizando exclusivamente observaciones anteriores a ella.

De esta forma, ninguna característica incorporará valores de demanda correspondientes a la hora que se desea predecir ni a horas posteriores.

In [4]:
# Crear retardos respecto a la hora objetivo
df["lag_1"] = df["demand_mw"]
df["lag_2"] = df["demand_mw"].shift(1)
df["lag_24"] = df["demand_mw"].shift(23)
df["lag_168"] = df["demand_mw"].shift(167)

# Comprobar los resultados
display(
    df[
        [
            "target_datetime_utc",
            "target_next_hour",
            "lag_1",
            "lag_2",
            "lag_24",
            "lag_168"
        ]
    ].tail(10)
)

print("\nValores nulos por retardo:")
print(
    df[
        ["lag_1", "lag_2", "lag_24", "lag_168"]
    ].isna().sum()
)


,target_datetime_utc,target_next_hour,lag_1,lag_2,lag_24,lag_168
8774,2024-12-31 15:00:00+00:00,27285.0,27261.0,27597.0,28500.0,25986.0
8775,2024-12-31 16:00:00+00:00,28010.0,27285.0,27261.0,29365.0,26759.0
8776,2024-12-31 17:00:00+00:00,30274.0,28010.0,27285.0,31293.0,29019.0
8777,2024-12-31 18:00:00+00:00,31225.0,30274.0,28010.0,32151.0,29433.0
8778,2024-12-31 19:00:00+00:00,30494.0,31225.0,30274.0,32701.0,28433.0
8779,2024-12-31 20:00:00+00:00,29005.0,30494.0,31225.0,32762.0,26863.0
8780,2024-12-31 21:00:00+00:00,26133.0,29005.0,30494.0,30838.0,24045.0
8781,2024-12-31 22:00:00+00:00,24316.0,26133.0,29005.0,28024.0,22511.0
8782,2024-12-31 23:00:00+00:00,23101.0,24316.0,26133.0,25394.0,21778.0
8783,2025-01-01 00:00:00+00:00,NaN,23101.0,24316.0,22886.0,20959.0



Valores nulos por retardo:
lag_1        0
lag_2        1
lag_24      23
lag_168    167
dtype: int64


## 6. Creación de estadísticas móviles

### Objetivo

Capturar tendencias recientes y variaciones en la demanda eléctrica mediante estadísticas calculadas sobre ventanas temporales.

### Variables generadas

- `rolling_mean_24`: demanda media de las últimas 24 horas conocidas.
- `rolling_std_24`: desviación estándar de las últimas 24 horas conocidas.
- `rolling_mean_168`: demanda media de las últimas 168 horas conocidas.

Estas variables permiten representar tanto el nivel reciente de demanda como su variabilidad.

### Prevención del Data Leakage

Las ventanas móviles incluirán únicamente observaciones disponibles hasta la hora actual, excluyendo la hora que se desea predecir.

Las estadísticas se calcularán de forma retrospectiva, sin utilizar información futura.

In [5]:
# Media de las últimas 24 horas conocidas
df["rolling_mean_24"] = (
    df["demand_mw"]
    .rolling(window=24)
    .mean()
)

# Desviación estándar de las últimas 24 horas
df["rolling_std_24"] = (
    df["demand_mw"]
    .rolling(window=24)
    .std()
)

# Media de la última semana
df["rolling_mean_168"] = (
    df["demand_mw"]
    .rolling(window=168)
    .mean()
)

# Mostrar las nuevas características
display(
    df[
        [
            "datetime_utc",
            "demand_mw",
            "rolling_mean_24",
            "rolling_std_24",
            "rolling_mean_168"
        ]
    ].tail(10)
)

print("\nValores nulos:")
print(
    df[
        [
            "rolling_mean_24",
            "rolling_std_24",
            "rolling_mean_168"
        ]
    ].isna().sum()
)


,datetime_utc,demand_mw,rolling_mean_24,rolling_std_24,rolling_mean_168
8774,2024-12-31 14:00:00+00:00,27261.0,26807.166667,4066.873435,24935.654762
8775,2024-12-31 15:00:00+00:00,27285.0,26756.541667,4052.421176,24943.386905
8776,2024-12-31 16:00:00+00:00,28010.0,26700.083333,4023.838217,24950.833333
8777,2024-12-31 17:00:00+00:00,30274.0,26657.625000,3978.387447,24958.303571
8778,2024-12-31 18:00:00+00:00,31225.0,26619.041667,3926.952857,24968.970238
8779,2024-12-31 19:00:00+00:00,30494.0,26527.083333,3802.197213,24981.238095
8780,2024-12-31 20:00:00+00:00,29005.0,26370.541667,3606.648100,24993.988095
8781,2024-12-31 21:00:00+00:00,26133.0,26174.500000,3478.866682,25006.416667
8782,2024-12-31 22:00:00+00:00,24316.0,26020.000000,3475.493672,25017.160714
8783,2024-12-31 23:00:00+00:00,23101.0,25924.458333,3524.620867,25025.035714



Valores nulos:
rolling_mean_24      23
rolling_std_24       23
rolling_mean_168    167
dtype: int64


## 7. Preparación del conjunto de datos para Machine Learning

Una vez generadas las características temporales, históricas y estadísticas, se seleccionan las variables que utilizarán los modelos predictivos.

Los primeros registros de la serie no disponen de suficiente información histórica para calcular todas las características. Asimismo, la última observación carece de un valor objetivo conocido.

Por ello, se eliminarán únicamente las filas que presenten valores ausentes en las variables seleccionadas o en la variable objetivo.

El dataset resultante se almacenará en formato Parquet para facilitar su reutilización durante la fase de entrenamiento y evaluación de modelos.

In [6]:
from pathlib import Path

# Seleccionar características predictoras
feature_columns = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168",
]

target_column = "target_next_hour"

# Seleccionar columnas relevantes
model_df = df[
    [
        "target_datetime_utc",
        *feature_columns,
        target_column
    ]
].copy()

# Eliminar filas sin información suficiente
model_df = model_df.dropna().reset_index(drop=True)

# Comprobar resultado
print("Dimensiones finales:", model_df.shape)
print("\nValores nulos:", model_df.isna().sum().sum())

display(model_df.head())

# Guardar dataset
output_path = Path(
    "../data/processed/features_spain_2024.parquet"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

model_df.to_parquet(output_path, index=False)

print("\nDataset guardado en:", output_path)


Dimensiones finales: (8616, 13)

Valores nulos: 0


,target_datetime_utc,hour,day_of_week,month,is_weekend,lag_1,lag_2,lag_24,lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168,target_next_hour
0,2024-01-08 00:00:00+00:00,1,0,1,0,24806.0,27351.0,22195.0,21378.0,24656.375000,3884.662571,25862.232143,22810.0
1,2024-01-08 01:00:00+00:00,2,0,1,0,22810.0,24806.0,20702.0,20286.0,24682.000000,3869.719965,25870.755952,21687.0
2,2024-01-08 02:00:00+00:00,3,0,1,0,21687.0,22810.0,19803.0,19074.0,24723.041667,3830.700066,25879.095238,21283.0
3,2024-01-08 03:00:00+00:00,4,0,1,0,21283.0,21687.0,19307.0,18261.0,24784.708333,3759.300692,25892.244048,21154.0
4,2024-01-08 04:00:00+00:00,5,0,1,0,21154.0,21283.0,19281.0,17899.0,24861.666667,3659.879253,25909.464286,21854.0



Dataset guardado en: ../data/processed/features_spain_2024.parquet


In [7]:
print("Dimensiones:", model_df.shape)
print("Valores nulos:", model_df.isna().sum().sum())

print("\nPrimera fecha objetivo:")
print(model_df["target_datetime_utc"].min())

print("\nÚltima fecha objetivo:")
print(model_df["target_datetime_utc"].max())

print("\nColumnas:")
print(model_df.columns.tolist())


Dimensiones: (8616, 13)
Valores nulos: 0

Primera fecha objetivo:
2024-01-08 00:00:00+00:00

Última fecha objetivo:
2024-12-31 23:00:00+00:00

Columnas:
['target_datetime_utc', 'hour', 'day_of_week', 'month', 'is_weekend', 'lag_1', 'lag_2', 'lag_24', 'lag_168', 'rolling_mean_24', 'rolling_std_24', 'rolling_mean_168', 'target_next_hour']
